# Week 8 — 走動式預測與回測完整性

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 實作時間式 train/test 切分與走動式評估。
- 建立預測模型並與 naive baseline 比較。
- 套用交易成本，比較 gross 與 net 績效。
- 刻意展示一個 leaked 模型，並說明為何其結果無效。

## 預估學習時間

約 10–12 小時。

## 先備概念

- Week 7 的時間序列
- Week 5 的迴歸

## 外部學習資源

- [Forecasting: Principles and Practice, the Pythonic Way](https://otexts.com/fpppy/)
- [Penn State STAT 510 Applied Time Series Analysis](https://online.stat.psu.edu/stat510/)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

## 概念說明

本週把前七週整合成一個**正確的**研究流程。核心原則：

1. **時間式切分**：訓練集一定在測試集之前。
2. **走動式評估**：expanding（錨定起點、視窗變長）或 rolling（固定長度、向前滑動）。
3. **無 leakage**：時間 $t$ 的特徵只能用 $t$ 以前的資訊；由訊號決定的部位必須**正確地往後位移**才能對應未來報酬。
4. **交易成本**：比較 gross 與 net；net 才是誠實的結果。
5. **baseline**：贏不過 naive baseline 的模型沒有價值。

> 本 notebook 是**方法論訓練**，不是可投資策略。所有結果都不代表真實可獲利性。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_prices
from quant_math_roadmap.finance.returns import simple_returns
from quant_math_roadmap.time_series.splits import (
    expanding_window_splits, train_test_split_time,
)
from quant_math_roadmap.time_series.forecasting import (
    forecast_error_metrics, historical_mean_forecast, zero_forecast,
)
from quant_math_roadmap.backtesting.engine import run_backtest, buy_and_hold_benchmark
from quant_math_roadmap.backtesting.leakage_checks import (
    assert_no_lookahead, leaked_strategy_returns, signal_to_positions,
)
from quant_math_roadmap.backtesting.costs import cost_summary

config = SyntheticConfig(n_assets=1, n_periods=900, seed=33)
prices = generate_correlated_prices(config).iloc[:, 0]
returns = simple_returns(prices)
print('報酬序列長度:', len(returns))

### 時間式 train/test 切分

In [ ]:
split = train_test_split_time(len(returns), test_size=0.3)
train = returns.iloc[split.train_index]
test = returns.iloc[split.test_index]
print(f'訓練集: {len(train)} 期 | 測試集: {len(test)} 期')
print('訓練集最後一天 <', '測試集第一天:',
      train.index[-1] < test.index[0])

### 預測模型 vs naive baseline

In [ ]:
# 用走動式 expanding window 做誠實的逐期預測
predictions, actuals, baseline_mean, baseline_zero = [], [], [], []
for sp in expanding_window_splits(len(returns), initial_train_size=400,
                                  test_size=1):
    tr = returns.iloc[sp.train_index]
    te = returns.iloc[sp.test_index]
    # 簡單線性 lag 預測：用最近一期報酬乘上訓練集的 lag-1 自相關
    lag1 = tr.autocorr(lag=1)
    pred = lag1 * tr.iloc[-1]
    predictions.append(pred)
    actuals.append(te.iloc[0])
    baseline_mean.append(historical_mean_forecast(tr))
    baseline_zero.append(zero_forecast(tr))

actual_s = pd.Series(actuals)
print('lag 預測   :', forecast_error_metrics(actual_s, pd.Series(predictions)))
print('歷史均值   :', forecast_error_metrics(actual_s, pd.Series(baseline_mean)))
print('naive 零   :', forecast_error_metrics(actual_s, pd.Series(baseline_zero)))

在合成的（近乎白噪音）報酬上，預測模型**通常贏不過** naive baseline。這是健康的結果：它誠實反映了「報酬很難預測」。

### 把訊號轉成部位（正確位移以避免 leakage）

In [ ]:
# 訊號：昨天報酬的正負號。部位必須往後位移 1 期才能交易。
raw_signal = np.sign(returns)
positions = signal_to_positions(raw_signal, lag=1)
print('前 5 個訊號:', raw_signal.head().to_list())
print('前 5 個部位:', positions.head().to_list())
print('部位是位移後的訊號 — 第一天為 0（沒有可用資訊）。')

### Gross vs net：交易成本的影響

In [ ]:
result = run_backtest(raw_signal, returns, signal_lag=1,
                      cost_per_unit_turnover=0.0005)
summary = result.summary()
for k, v in summary.items():
    print(f'{k}: {v:.6f}')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(result.gross_equity.index, result.gross_equity.values,
        label='gross（未計成本）')
ax.plot(result.net_equity.index, result.net_equity.values,
        label='net（已計成本）')
bnh = buy_and_hold_benchmark(returns)
ax.plot(bnh.index, bnh.values, label='buy-and-hold 基準', linestyle='--')
ax.set_title('回測權益曲線：gross vs net vs 基準')
ax.set_xlabel('日期')
ax.set_ylabel('權益（起始 = 1）')
ax.legend()
plt.show()

交易成本把 gross 與 net 拉開明顯差距。**只看 gross 的回測是不誠實的。**

### 刻意展示 data leakage（無效示範）

下面這個實驗**故意作弊**：它用「**當期**報酬的正負號」當部位——這需要知道未來。它必然每期都賺，績效荒謬地好。

> **這個結果絕對不能被當成策略。** 它存在的唯一目的，是讓你認得leakage 長什麼樣子。

In [ ]:
leaked = leaked_strategy_returns(returns)
leaked_equity = (1 + leaked).cumprod()
honest_equity = result.net_equity

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(leaked_equity.index, leaked_equity.values,
        label='【無效】leaked 模型（偷看未來）')
ax.plot(honest_equity.index, honest_equity.values,
        label='誠實回測（net）')
ax.set_title('Leakage 示範：荒謬的曲線 = 警訊，不是策略')
ax.set_xlabel('日期')
ax.set_ylabel('權益（起始 = 1）')
ax.legend()
plt.show()
print(f'leaked 總報酬 = {(leaked_equity.iloc[-1] - 1):.2%}  <-- 不可能、無效')
print(f'誠實 net 總報酬 = {(honest_equity.iloc[-1] - 1):.2%}')

In [ ]:
# leakage 自動檢查：把當期報酬當特徵會被偵測出來
try:
    assert_no_lookahead(returns, returns, name='當期報酬當特徵')
    print('未偵測到 leakage')
except ValueError as exc:
    print('偵測到 leakage:', exc)

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 用自己的話解釋 expanding window 與 rolling window 的差別。
2. 為什麼由訊號決定的部位一定要往後位移？
3. 什麼是 survivorship bias？它為何會讓回測過度樂觀？

### 應用練習

In [ ]:
# 應用練習 1：把交易成本從 5 bps 提高到 20 bps，比較 net 總報酬。
high_cost = None  # TODO: run_backtest(raw_signal, returns, cost_per_unit_turnover=0.002)
if high_cost is not None:
    print('20 bps net 總報酬:', high_cost.summary()['total_net_return'])

In [ ]:
# 應用練習 2：用 cost_summary 比較 gross 與 net 的總報酬與成本拖累。
cs = None  # TODO: cost_summary(result.gross_returns, result.net_returns)
if cs is not None:
    print(cs)

### 反思問題

1. 回顧 Week 1–7。一個看起來很賺的回測，可能在哪些環節偷偷引入了 leakage、過度配適或多重檢定的問題？請至少列出三點，並對照 [`docs/common_backtesting_mistakes.md`](../docs/common_backtesting_mistakes.md)。

## 常見錯誤

- **用隨機切分而非時間式切分。**
- **部位沒有往後位移，等於偷看未來（look-ahead bias）。**
- **只報告 gross 績效，忽略交易成本與週轉。**
- **沒有和 naive baseline（零報酬、歷史均值、buy-and-hold）比較。**
- **把刻意 leaked 的荒謬結果誤當成真實策略。**
- **把 rolling 特徵前期不足的 NaN 用未來資訊回填。**

## 完成本週後，你應該能做到什麼

- [ ] 能實作時間式切分與走動式評估。
- [ ] 能把預測模型和 naive baseline 做誠實比較。
- [ ] 能套用交易成本並比較 gross vs net。
- [ ] 能辨識 leakage，並說明為何其結果無效。
- [ ] 能完成一個無 leakage、含成本、可重現的小型回測流程。

## 結語

完成八週後，你已經能從「複習數學基礎」走到「一個小而正確、防 leakage 的量化研究流程」。請記得本路線圖的核心信念：

**先正確評估，再追求精緻模型；先可重現，再談績效。**

本專案是教育與研究方法論訓練，**不是投資建議**，也**不宣稱**任何可獲利策略。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。